In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv


In [2]:
import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings

# 警告の抑制
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)


In [3]:

df_train = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv", index_col='id')
df_test = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv", index_col='id')
sub = pd.read_csv("/kaggle/input/playground-series-s6e2/sample_submission.csv")

# ターゲット変数のマッピング (Presence -> 1, Absence -> 0)
if df_train['Heart Disease'].dtype == 'object':
    df_train['Heart Disease'] = df_train['Heart Disease'].map({'Presence': 1, 'Absence': 0})


In [4]:

# 特徴量エンジニアリング 

def apply_domain_features(df):
    df_eng = df.copy()
    
    # --- 既存の特徴量作成 ---
    df_eng['RPP'] = df_eng['BP'] * df_eng['Max HR']
    df_eng['MaxHR_Expected'] = 220 - df_eng['Age']
    df_eng['MaxHR_Achieved_Ratio'] = df_eng['Max HR'] / df_eng['MaxHR_Expected']
    
    df_eng['BP_Risk'] = 0
    df_eng.loc[df_eng['BP'] >= 130, 'BP_Risk'] = 1
    df_eng.loc[df_eng['BP'] >= 140, 'BP_Risk'] = 2
    
    df_eng['Chol_Risk'] = 0
    df_eng.loc[df_eng['Cholesterol'] >= 200, 'Chol_Risk'] = 1
    df_eng.loc[df_eng['Cholesterol'] >= 240, 'Chol_Risk'] = 2
    
    df_eng['Age_BP_Risk'] = df_eng['Age'] * df_eng['BP']
    df_eng['Vessels_ST_Interaction'] = df_eng['Number of vessels fluro'] * df_eng['ST depression']

    return df_eng

print("Applying feature engineering...")
X = apply_domain_features(df_train.drop(['Heart Disease'], axis=1))
y = df_train['Heart Disease']
X_test = apply_domain_features(df_test)

# カテゴリカル変数の処理 (Label Encoding)
cat_cols = ['Sex', 'Chest pain type', 'FBS over 120', 'EKG results', 
            'Exercise angina', 'Slope of ST', 'Number of vessels fluro', 'Thallium',
            'BP_Risk', 'Chol_Risk']

# 訓練データとテストデータを結合してエンコーディング（未知のカテゴリ対策）
all_data = pd.concat([X, X_test])
for col in cat_cols:
    le = LabelEncoder()
    all_data[col] = le.fit_transform(all_data[col].astype(str))

X = all_data.iloc[:len(X)]
X_test = all_data.iloc[len(X):]


Applying feature engineering...


In [5]:

N_SPLITS = 5  # 交差検証の分割数
N_TRIALS = 20 # Optunaの試行回数 

class ModelTrainer:
    def __init__(self, model_name, X, y, X_test, cat_features):
        self.model_name = model_name
        self.X = X
        self.y = y
        self.X_test = X_test
        self.cat_features = cat_features
        self.best_params = None
        
    def objective(self, trial):
        # モデルごとの探索パラメータ定義
        if self.model_name == 'lgbm':
            params = {
                'objective': 'binary',
                'metric': 'auc',
                'verbosity': -1,
                'n_estimators': 1000,
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
                'num_leaves': trial.suggest_int('num_leaves', 20, 100),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            }
        elif self.model_name == 'xgb':
            params = {
                'objective': 'binary:logistic',
                'eval_metric': 'auc',
                'n_estimators': 1000,
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
                'enable_categorical': True  # XGBoostのカテゴリカル対応有効化
            }
        elif self.model_name == 'cat':
            params = {
                'loss_function': 'Logloss',
                'eval_metric': 'AUC',
                'iterations': 1000,
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
                'depth': trial.suggest_int('depth', 4, 10),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
                'random_strength': trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
                'border_count': 254,
                'verbose': False,
                'task_type': 'GPU' 
            }

        # 高速化のため、分割数を減らして探索
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        aucs = []
        
        for train_idx, val_idx in skf.split(self.X, self.y):
            X_tr, X_val = self.X.iloc[train_idx], self.X.iloc[val_idx]
            y_tr, y_val = self.y.iloc[train_idx], self.y.iloc[val_idx]
            
            if self.model_name == 'lgbm':
                model = lgb.LGBMClassifier(**params, random_state=42)
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
                          callbacks=[lgb.early_stopping(50, verbose=False)])
            elif self.model_name == 'xgb':
                model = xgb.XGBClassifier(**params, random_state=42, tree_method='hist')
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False, 
                          #early_stopping_rounds=50
                         )
            elif self.model_name == 'cat':
                model = CatBoostClassifier(**params, random_seed=42)
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
                          cat_features=self.cat_features, verbose=False, early_stopping_rounds=50)
                
            pred = model.predict_proba(X_val)[:, 1]
            aucs.append(roc_auc_score(y_val, pred))
            
        return np.mean(aucs)

    def tune_and_train(self):
        print(f"--- Tuning {self.model_name.upper()} ---")
        study = optuna.create_study(direction='maximize')
        study.optimize(self.objective, n_trials=N_TRIALS)
        self.best_params = study.best_params
        print(f"Best params: {self.best_params}")
        
        # 最適パラメータで本番学習 (5-Fold)
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
        oof_preds = np.zeros(len(self.X))
        test_preds = np.zeros(len(self.X_test))
        
        print(f"--- Training {self.model_name.upper()} with Best Params ---")
        
        # モデル共通の固定パラメータを追加
        if self.model_name == 'lgbm':
            self.best_params.update({'n_estimators': 10000, 'objective': 'binary', 'metric': 'auc', 'verbosity': -1})
        elif self.model_name == 'xgb':
            self.best_params.update({'n_estimators': 10000, 'objective': 'binary:logistic', 'eval_metric': 'auc', 'enable_categorical': True})
        elif self.model_name == 'cat':
            self.best_params.update({'iterations': 10000, 'loss_function': 'Logloss', 'eval_metric': 'AUC', 'verbose': False})

        for fold, (train_idx, val_idx) in enumerate(skf.split(self.X, self.y)):
            X_tr, X_val = self.X.iloc[train_idx], self.X.iloc[val_idx]
            y_tr, y_val = self.y.iloc[train_idx], self.y.iloc[val_idx]
            
            if self.model_name == 'lgbm':
                model = lgb.LGBMClassifier(**self.best_params, random_state=42)
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
                          callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
            elif self.model_name == 'xgb':
                model = xgb.XGBClassifier(**self.best_params, random_state=42, tree_method='hist')
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False, 
                          #early_stopping_rounds=100
                         )
            elif self.model_name == 'cat':
                model = CatBoostClassifier(**self.best_params, random_seed=42)
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
                          cat_features=self.cat_features, verbose=False, early_stopping_rounds=100)
            
            val_pred = model.predict_proba(X_val)[:, 1]
            oof_preds[val_idx] = val_pred
            test_preds += model.predict_proba(self.X_test)[:, 1] / N_SPLITS
            
            score = roc_auc_score(y_val, val_pred)
            print(f"Fold {fold+1} AUC: {score:.5f}")
            
        print(f"{self.model_name.upper()} Overall AUC: {roc_auc_score(self.y, oof_preds):.5f}")
        return oof_preds, test_preds




In [6]:

cat_indices = [X.columns.get_loc(c) for c in cat_cols if c in X.columns]

# LightGBM
trainer_lgb = ModelTrainer('lgbm', X, y, X_test, cat_cols)
oof_lgb, pred_lgb = trainer_lgb.tune_and_train()

# XGBoost
trainer_xgb = ModelTrainer('xgb', X, y, X_test, cat_cols)
oof_xgb, pred_xgb = trainer_xgb.tune_and_train()

trainer_cat = ModelTrainer('cat', X, y, X_test, cat_indices)
oof_cat, pred_cat = trainer_cat.tune_and_train()


--- Tuning LGBM ---
Best params: {'learning_rate': 0.04790139207494703, 'num_leaves': 48, 'max_depth': 4, 'min_child_samples': 82, 'subsample': 0.8546299271266478, 'colsample_bytree': 0.6030376538260056, 'reg_alpha': 0.0007943271108072904, 'reg_lambda': 3.194188167277616e-06}
--- Training LGBM with Best Params ---
Fold 1 AUC: 0.95569
Fold 2 AUC: 0.95465
Fold 3 AUC: 0.95551
Fold 4 AUC: 0.95505
Fold 5 AUC: 0.95592
LGBM Overall AUC: 0.95536
--- Tuning XGB ---
Best params: {'learning_rate': 0.079958209465731, 'max_depth': 3, 'min_child_weight': 10, 'subsample': 0.9647451358321659, 'colsample_bytree': 0.6855832040514374, 'reg_alpha': 4.5713881148053683e-07, 'reg_lambda': 4.257474072869499e-06}
--- Training XGB with Best Params ---
Fold 1 AUC: 0.95478
Fold 2 AUC: 0.95361
Fold 3 AUC: 0.95459
Fold 4 AUC: 0.95414
Fold 5 AUC: 0.95513
XGB Overall AUC: 0.95445
--- Tuning CAT ---


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric perio

Best params: {'learning_rate': 0.09560809958465857, 'depth': 4, 'l2_leaf_reg': 1.4005286560190342, 'random_strength': 0.007477257823411844}
--- Training CAT with Best Params ---
Fold 1 AUC: 0.95586
Fold 2 AUC: 0.95474
Fold 3 AUC: 0.95561
Fold 4 AUC: 0.95514
Fold 5 AUC: 0.95599
CAT Overall AUC: 0.95547


In [7]:
# AUCスコアが高いモデルに少し重みを置く
score_lgb = roc_auc_score(y, oof_lgb)
score_xgb = roc_auc_score(y, oof_xgb)
score_cat = roc_auc_score(y, oof_cat)
print(f"OOF Scores - LGB: {score_lgb:.5f}, XGB: {score_xgb:.5f}, CAT: {score_cat:.5f}")

# 重み計算 (合計1になるように正規化)
total_score = score_lgb + score_xgb + score_cat
w_lgb = score_lgb / total_score
w_xgb = score_xgb / total_score
w_cat = score_cat / total_score

print(f"Weights - LGB: {w_lgb:.3f}, XGB: {w_xgb:.3f}, CAT: {w_cat:.3f}")

final_oof = (oof_lgb * w_lgb) + (oof_xgb * w_xgb) + (oof_cat * w_cat)
final_score = roc_auc_score(y, final_oof)
print(f"Ensemble OOF AUC: {final_score:.5f}")

# テストデータ予測の作成
final_pred = (pred_lgb * w_lgb) + (pred_xgb * w_xgb) + (pred_cat * w_cat)


OOF Scores - LGB: 0.95536, XGB: 0.95445, CAT: 0.95547
Weights - LGB: 0.333, XGB: 0.333, CAT: 0.333
Ensemble OOF AUC: 0.95538


In [8]:
sub['Heart Disease'] = final_pred
sub.to_csv('submission_ensemble.csv', index=False)